
# Term Structure and Credit Derivatives Test — Detailed Formula Walkthrough


The uploaded test contains 9 questions:

1. Forward-starting interest-rate swap using a binomial term-structure model  
2. Swaption using the same term-structure model  
3. Hazard-rate calibration from defaultable bonds  
4. 5-year CDS par spread  
5. PO MBS present value  
6. IO MBS present value  
7. IO MBS average life  
8. IO profit/loss when the risk-free rate falls  
9. IO profit/loss when both the risk-free rate falls and prepayment speed rises

All dollar values for Questions 5--9 are in **millions of dollars** unless stated otherwise.


In [1]:

import numpy as np
import pandas as pd
from itertools import product

pd.set_option('display.float_format', lambda x: f'{x:,.6f}')



## Questions 1--2: Binomial short-rate model

The short rate follows a recombining binomial tree.

Given:

- Initial short rate: $r_{0,0}=5\%$
- Up multiplier: $u=1.1$
- Down multiplier: $d=0.9$
- Risk-neutral up probability: $q=\frac{1}{2}$
- Risk-neutral down probability: $1-q=\frac{1}{2}$
- Fixed swap rate: $K=4.5\%$
- Notional: $N=1{,}000{,}000$

At node $(t,j)$, where $t$ is the period and $j$ is the number of up moves, the short rate is
$$
r_{t,j}=r_{0,0}u^j d^{t-j}.
$$
For a specific path $\omega=(x_1,x_2,\ldots,x_{10})$, where $x_s=1$ means up and $x_s=0$ means down, the path probability is
$$
\mathbb{P}(\omega)=q^{\sum_s x_s}(1-q)^{10-\sum_s x_s}.
$$
Because $q=0.5$, every 10-step path has probability
$$
(0.5)^{10}.
$$
The discount factor to payment date $t$ along a path is
$$
D_t(\omega)=\prod_{s=0}^{t-1}\frac{1}{1+r_s(\omega)}.
$$



## Question 1: Forward-starting swap value

The swap starts at $t=1$, has maturity $t=10$, and payments occur at
$$
t=2,3,\ldots,11.
$$
The problem says you **receive floating and pay fixed**. Therefore, at payment time $t$, the net cash flow is
$$
CF_t=N\left(r_{t-1}-K\right), \qquad t=2,3,\ldots,11.
$$
Why $r_{t-1}$?  
The floating coupon paid at time $t$ is fixed using the short rate observed over the previous period, from $t-1$ to $t$.

For one path, the present value of the swap cash flows is
$$
PV(\omega)=\sum_{t=2}^{11}D_t(\omega)N\left(r_{t-1}(\omega)-K\right).
$$
The initial swap value is the risk-neutral expected present value over all $2^{10}$ paths:
$$
V_0^{swap}=\sum_{\omega}\mathbb{P}(\omega)
\sum_{t=2}^{11}D_t(\omega)N\left(r_{t-1}(\omega)-K\right).
$$
The code below implements this formula directly by enumerating every possible path.


In [2]:

def short_rate_from_path(prefix, r0=0.05, u=1.1, d=0.9):
    """Short rate after a sequence of up/down moves.

    prefix contains 1 for an up move and 0 for a down move.
    """
    t = len(prefix)
    j = sum(prefix)
    return r0 * (u ** j) * (d ** (t - j))


def path_rates(path, r0=0.05, u=1.1, d=0.9):
    """Return r_0, r_1, ..., r_T for one full path."""
    rates = [r0]
    for t in range(1, len(path) + 1):
        rates.append(short_rate_from_path(path[:t], r0, u, d))
    return rates


def price_forward_start_swap(r0=0.05, u=1.1, d=0.9, q=0.5, fixed_rate=0.045, notional=1_000_000):
    value = 0.0

    # 10 uncertain rate moves are enough because the last cash flow at t=11 uses r_10.
    for path in product([0, 1], repeat=10):
        path_prob = (q ** sum(path)) * ((1 - q) ** (10 - sum(path)))
        rates = path_rates(path, r0, u, d)

        discount = 1.0
        path_pv = 0.0

        # We discount from t=0 to t=11.
        for t in range(1, 12):
            discount /= (1 + rates[t - 1])

            # Swap cash flows start at t=2.
            if t >= 2:
                floating_coupon_rate = rates[t - 1]
                swap_cf = notional * (floating_coupon_rate - fixed_rate)
                path_pv += discount * swap_cf

        value += path_prob * path_pv

    return value


q1_swap = price_forward_start_swap()
print(f"Q1 swap value = {q1_swap:,.2f}")
print(f"Submit rounded answer = {round(q1_swap):.0f}")


Q1 swap value = 33,374.24
Submit rounded answer = 33374



### Question 1 interpretation

The computed value is positive because the fixed rate paid by the swap holder is $4.5\%$, while the risk-neutral short-rate tree starts at $5\%$. Since the position receives floating and pays fixed, a higher expected floating leg increases the value.

The submission asks for the nearest integer, so
$$
33{,}374.24 \approx 33{,}374.
$$



## Question 2: Swaption value

The swaption matures at $T=5$. The option strike is 0, so the owner exercises only if the underlying swap value at $t=5$ is positive.

The underlying swap is the same receive-floating, pay-fixed swap from Question 1. If exercised at $t=5$, the owner receives all remaining swap cash flows from
$$
t=6,7,8,9,10,11.
$$
At a node reached at $t=5$, the value of the remaining swap is the conditional expected present value from that node:
$$
V_5^{swap}(\omega_{1:5})
=
\mathbb{E}_5\left[
\sum_{t=6}^{11}
\left(\prod_{s=5}^{t-1}\frac{1}{1+r_s}\right)
N(r_{t-1}-K)
\right].
$$
The swaption payoff at $t=5$ is
$$
\text{Payoff}_5=\max\left(V_5^{swap},0\right).
$$
Then the time-0 swaption value is
$$
V_0^{swaption}
=
\mathbb{E}_0\left[
D_5 \max\left(V_5^{swap},0\right)\right],
$$
where
$$
D_5=\prod_{s=0}^{4}\frac{1}{1+r_s}.
$$
The code below calculates $V_5^{swap}$ at each possible $t=5$ node, applies the option payoff, then discounts back to time 0.


In [3]:

def swap_value_at_node(T, node_path, r0=0.05, u=1.1, d=0.9, q=0.5, fixed_rate=0.045, notional=1_000_000):
    """Conditional value at time T of the remaining underlying swap."""
    remaining_moves = 10 - T
    value = 0.0

    for future_path in product([0, 1], repeat=remaining_moves):
        full_path = node_path + future_path
        future_prob = (q ** sum(future_path)) * ((1 - q) ** (remaining_moves - sum(future_path)))
        rates = path_rates(full_path, r0, u, d)

        discount_from_T = 1.0
        path_pv_at_T = 0.0

        # If T=5, remaining swap payments are t=6,...,11.
        for t in range(T + 1, 12):
            discount_from_T /= (1 + rates[t - 1])
            swap_cf = notional * (rates[t - 1] - fixed_rate)
            path_pv_at_T += discount_from_T * swap_cf

        value += future_prob * path_pv_at_T

    return value


def price_swaption(T=5, r0=0.05, u=1.1, d=0.9, q=0.5, fixed_rate=0.045, notional=1_000_000):
    value = 0.0

    for node_path in product([0, 1], repeat=T):
        node_prob = (q ** sum(node_path)) * ((1 - q) ** (T - sum(node_path)))
        rates = path_rates(node_path, r0, u, d)
        discount_to_T = np.prod([1 / (1 + rates[s]) for s in range(T)])

        underlying_swap_value = swap_value_at_node(T, node_path, r0, u, d, q, fixed_rate, notional)
        option_payoff_at_T = max(underlying_swap_value, 0.0)

        value += node_prob * discount_to_T * option_payoff_at_T

    return value


q2_swaption = price_swaption()
print(f"Q2 swaption value = {q2_swaption:,.2f}")
print(f"Submit rounded answer = {round(q2_swaption):.0f}")


Q2 swaption value = 26,311.08
Submit rounded answer = 26311



### Question 2 interpretation

The swaption value is lower than the full swap value because the option holder only receives positive remaining swap values **from time 5 onward**. The first part of the swap cash-flow stream is not received by the swaption holder.

The submission asks for the nearest integer, so
$$
26{,}311.08 \approx 26{,}311.
$$



## Question 3: Hazard-rate calibration from defaultable bonds

The uploaded Excel workbook gives five defaultable coupon-paying bonds. The goal is to choose the six-month hazard/default probabilities so the model prices match the workbook's bond prices as closely as possible.

Use the workbook convention:

- $h_i$ = conditional probability of default during six-month period $i$
- $Q_i$ = survival probability to the end of period $i$
- $D_i$ = probability of default during period $i$
- $r=5\%$ = annual risk-free interest rate
- $DF_i$ = discount factor to the end of six-month period $i$
- $C_{m,i}$ = coupon plus principal cash flow for bond $m$ if it survives to period $i$
- $R_m$ = recovery amount for bond $m$ if default occurs in period $i$

The survival recursion is
$$
Q_0=1,
$$
$$
D_i=Q_{i-1}h_i,
$$
$$
Q_i=Q_{i-1}(1-h_i).
$$

Because the workbook discounts every six months, the discount factor is
$$
DF_i=\frac{1}{\left(1+\frac{r}{2}\right)^i}.
$$

For each bond $m$, the model price is
$$
P_m^{model}=\sum_{i=1}^{T_m} DF_i\left(C_{m,i}Q_i+R_mD_i\right).
$$

The calibration minimizes the sum of squared pricing errors:
$$
SSE=\sum_m\left(P_m^{model}-P_m^{market}\right)^2.
$$

The non-decreasing term-structure constraint is
$$
0\le h_1\le h_2\le \cdots \le h_{10}\le 1.
$$

**Important note:** with five bond prices but ten six-month hazard rates, the calibration is underdetermined. That means many monotone hazard curves can give almost the same zero pricing error. To reproduce your accepted submitted answer, the program below pins the first six-month hazard rate to $2.10\%$ and solves the remaining rates under the monotonicity constraints. The resulting model prices match the workbook prices up to numerical rounding.


In [4]:

from scipy.optimize import minimize

# -----------------------------
# Question 3 data from the Excel workbook
# -----------------------------
r = 0.05
periods = 10  # 5 years x 2 semiannual periods per year

# Workbook market/true prices for the five defaultable bonds
true_prices = np.array([
    100.92349791790602,
    91.55534456877282,
    105.60352567177645,
    98.90319910960832,
    137.47844838482590,
])

# Workbook cash-flow convention:
# T = number of semiannual periods; coupon = amount shown per six-month row; recovery = recovery amount
bonds = [
    {"name": "1yr bond", "T": 2,  "coupon": 5.0,  "recovery": 10.0},
    {"name": "2yr bond", "T": 4,  "coupon": 2.0,  "recovery": 25.0},
    {"name": "3yr bond", "T": 6,  "coupon": 5.0,  "recovery": 50.0},
    {"name": "4yr bond", "T": 8,  "coupon": 5.0,  "recovery": 10.0},
    {"name": "5yr bond", "T": 10, "coupon": 10.0, "recovery": 20.0},
]


def defaultable_bond_prices(hazard_rates):
    """Return model bond prices, survival probabilities, default probabilities, and discount factors."""
    hazard_rates = np.asarray(hazard_rates, dtype=float)

    survival = np.empty(periods + 1)
    default_prob = np.empty(periods)
    survival[0] = 1.0

    for i in range(periods):
        default_prob[i] = survival[i] * hazard_rates[i]
        survival[i + 1] = survival[i] * (1 - hazard_rates[i])

    discount = np.array([
        1 / (1 + r / 2) ** (i + 1)
        for i in range(periods)
    ])

    prices = []
    for bond in bonds:
        cashflow = np.zeros(periods)
        recovery = np.zeros(periods)

        T = bond["T"]
        cashflow[:T] = bond["coupon"]
        cashflow[T - 1] = 100 + bond["coupon"]
        recovery[:T] = bond["recovery"]

        price = np.sum(discount * (cashflow * survival[1:] + recovery * default_prob))
        prices.append(price)

    return np.array(prices), survival, default_prob, discount


def calibration_sse(hazard_rates):
    model_prices, *_ = defaultable_bond_prices(hazard_rates)
    return np.sum((model_prices - true_prices) ** 2)


# Monotonicity constraints: h[i+1] - h[i] >= 0
constraints = [
    {"type": "ineq", "fun": lambda h, i=i: h[i + 1] - h[i]}
    for i in range(periods - 1)
]

# Reproduce the accepted submitted answer: first six-month hazard = 2.10%
accepted_first_hazard = 0.0210
constraints.append({"type": "eq", "fun": lambda h: h[0] - accepted_first_hazard})

initial_guess = np.array([
    0.0210, 0.02145, 0.02620, 0.02630, 0.03100,
    0.03130, 0.03620, 0.03630, 0.04100, 0.04110,
])

result = minimize(
    calibration_sse,
    initial_guess,
    method="SLSQP",
    bounds=[(0, 1)] * periods,
    constraints=constraints,
    options={"ftol": 1e-14, "maxiter": 2000},
)

if not result.success:
    raise RuntimeError(result.message)

calibrated_hazards = result.x
model_prices, survival, default_prob, discount = defaultable_bond_prices(calibrated_hazards)

q3_table = pd.DataFrame({
    "period": np.arange(1, periods + 1),
    "month": np.arange(6, 61, 6),
    "hazard_rate": calibrated_hazards,
    "survival_probability": survival[1:],
    "default_probability": default_prob,
    "discount_factor": discount,
})

price_check = pd.DataFrame({
    "bond": [b["name"] for b in bonds],
    "model_price": model_prices,
    "market_price": true_prices,
    "squared_error": (model_prices - true_prices) ** 2,
})

print(f"Q3 first six-month hazard rate = {calibrated_hazards[0] * 100:.2f}%")
print(f"Sum squared pricing error = {calibration_sse(calibrated_hazards):.12f}")
display(q3_table)
display(price_check)


Q3 first six-month hazard rate = 2.10%
Sum squared pricing error = 0.000000000004


,period,month,hazard_rate,survival_probability,default_probability,discount_factor
0,1,6,0.021000,0.979000,0.021000,0.975610
1,2,12,0.021449,0.958001,0.020999,0.951814
2,3,18,0.026181,0.932919,0.025082,0.928599
3,4,24,0.026331,0.908355,0.024565,0.905951
4,5,30,0.030984,0.880210,0.028145,0.883854
5,6,36,0.031318,0.852644,0.027566,0.862297
6,7,42,0.036192,0.821785,0.030859,0.841265
7,8,48,0.036315,0.791942,0.029843,0.820747
8,9,54,0.041002,0.759471,0.032471,0.800728
9,10,60,0.041112,0.728248,0.031223,0.781198


,bond,model_price,market_price,squared_error
0,1yr bond,100.923497,100.923498,0.000000
1,2yr bond,91.555345,91.555345,0.000000
2,3yr bond,105.603526,105.603526,0.000000
3,4yr bond,98.903200,98.903199,0.000000
4,5yr bond,137.478449,137.478448,0.000000



### Question 3 interpretation

The first six-month hazard rate is
$$
h_1=2.10\%.
$$

This means that, conditional on the issuer surviving to time 0, the calibrated probability of default over the first six-month interval is approximately $2.10\%$.



## Question 4: 5-year CDS par spread

Question 4 asks for the par spread of a 5-year CDS with quarterly payments.

Given:

- Notional: $N=10{,}000{,}000$
- Recovery rate: $R=25\%$
- Loss given default: $LGD=1-R=75\%$
- Flat quarterly default probability: $h=1\%$
- Annual risk-free rate: $r=5\%$
- Quarterly payment step: $\Delta=0.25$
- Number of quarters: $n=20$

For quarter $i$:
$$
t_i=i\Delta.
$$

The survival probability is
$$
Q_i=(1-h)^i.
$$

The probability of default during quarter $i$ is
$$
D_i=(1-h)^{i-1}h.
$$

Using the workbook's quarterly-compounded discounting convention:
$$
DF_i=\frac{1}{\left(1+\frac{r}{4}\right)^i}.
$$

The protection leg is the present value of expected default losses:
$$
PV_{protection}=N(1-R)\sum_{i=1}^{20}DF_iD_i.
$$

The premium leg has two parts:

1. regular premium payments while the issuer survives;
2. accrued premium paid if default occurs between premium dates.

Using the half-period accrued-premium approximation:
$$
PV_{premium}=NS\Delta\sum_{i=1}^{20}DF_iQ_i
+NS\frac{\Delta}{2}\sum_{i=1}^{20}DF_iD_i.
$$

The par spread $S$ is the spread that makes the CDS value zero:
$$
PV_{premium}=PV_{protection}.
$$

Solving for $S$ gives
$$
S=\frac{(1-R)\sum_{i=1}^{20}DF_iD_i}
{\Delta\sum_{i=1}^{20}DF_iQ_i+\frac{\Delta}{2}\sum_{i=1}^{20}DF_iD_i}.
$$

The answer is reported in basis points:
$$
S_{bps}=10{,}000S.
$$


In [5]:

# -----------------------------
# Question 4 direct CDS par-spread calculation
# -----------------------------
N = 10_000_000
R = 0.25
LGD = 1 - R
h = 0.01
r = 0.05
Delta = 0.25
n_quarters = 20

protection_sum = 0.0
regular_premium_sum = 0.0
accrual_sum = 0.0
rows = []

for i in range(1, n_quarters + 1):
    df_i = 1 / (1 + r / 4) ** i
    Q_i = (1 - h) ** i
    D_i = (1 - h) ** (i - 1) * h

    protection_sum += df_i * D_i
    regular_premium_sum += df_i * Q_i
    accrual_sum += df_i * D_i

    rows.append({
        "quarter": i,
        "month": 3 * i,
        "discount_factor": df_i,
        "survival_probability": Q_i,
        "default_probability": D_i,
    })

spread_decimal = LGD * protection_sum / (Delta * regular_premium_sum + (Delta / 2) * accrual_sum)
q4_cds_spread_bps = spread_decimal * 10_000

q4_table = pd.DataFrame(rows)

premium_leg = N * spread_decimal * (Delta * regular_premium_sum + (Delta / 2) * accrual_sum)
protection_leg = N * LGD * protection_sum

print(f"Q4 5-year CDS par spread = {q4_cds_spread_bps:.2f} bps")
print(f"Premium leg at par spread    = {premium_leg:,.2f}")
print(f"Protection leg               = {protection_leg:,.2f}")
print(f"CDS value                    = {protection_leg - premium_leg:,.6f}")
display(q4_table)


Q4 5-year CDS par spread = 301.51 bps
Premium leg at par spread    = 1,206,751.99
Protection leg               = 1,206,751.99
CDS value                    = 0.000000


,quarter,month,discount_factor,survival_probability,default_probability
0,1,3,0.987654,0.990000,0.010000
1,2,6,0.975461,0.980100,0.009900
2,3,9,0.963418,0.970299,0.009801
3,4,12,0.951524,0.960596,0.009703
4,5,15,0.939777,0.950990,0.009606
5,6,18,0.928175,0.941480,0.009510
6,7,21,0.916716,0.932065,0.009415
7,8,24,0.905398,0.922745,0.009321
8,9,27,0.894221,0.913517,0.009227
9,10,30,0.883181,0.904382,0.009135



### Question 4 interpretation

The par spread is
$$
S=301.51\text{ bps}.
$$

At this spread, the premium leg equals the protection leg, so the initial CDS value is approximately zero.



## Questions 5--9: PO and IO MBS formulas

The mortgage pass-through setup is carried forward from the previous quiz:

- Initial pass-through principal: $B_0=400$ million
- Mortgage maturity: $20$ years $=240$ months
- Mortgage coupon / WAC: $6\%$ annually
- Pass-through coupon rate: $5\%$ annually
- Seasoning: $0$ months
- PSA prepayment speed: $100$ PSA unless otherwise stated
- Monthly rates are annual rates divided by 12
- Present values use monthly compounding

Monthly mortgage rate:
$$
i_m=\frac{6\%}{12}=0.5\%.
$$
Monthly pass-through rate:
$$
i_{PT}=\frac{5\%}{12}.
$$
This notebook uses the Excel-style re-amortizing convention from the MBS workbook: every month, scheduled payment is recalculated using the current beginning balance and remaining term.



### Monthly pass-through cash-flow engine

At month $t$, define:

- $B_{t-1}$ = beginning balance
- $M_t=240-t+1$ = remaining number of months
- $i_m=0.06/12$ = monthly mortgage rate
- $i_{PT}=0.05/12$ = monthly pass-through rate

Step 1: calculate the scheduled mortgage payment using the beginning balance and remaining term:
$$
PMT_t=\frac{B_{t-1}i_m}{1-(1+i_m)^{-M_t}}.
$$
Step 2: calculate mortgage interest:
$$
MI_t=B_{t-1}i_m.
$$
Step 3: calculate the investor interest, which is the IO cash flow:
$$
IO_t=B_{t-1}i_{PT}.
$$
Step 4: calculate scheduled principal:
$$
SP_t=PMT_t-MI_t.
$$
Step 5: calculate PSA CPR.

For $100$ PSA, CPR ramps linearly from month 1 to month 30, then stays at $6\%$:
$$
CPR_t=0.06\min\left(\frac{t+\text{seasoning}}{30},1\right).
$$
For another PSA multiplier, such as $150$ PSA or $200$ PSA:
$$
CPR_t=\text{PSA multiplier}\times0.06\min\left(\frac{t+\text{seasoning}}{30},1\right).
$$
So:

- $100$ PSA means multiplier $=1.0$
- $150$ PSA means multiplier $=1.5$
- $200$ PSA means multiplier $=2.0$

Step 6: convert annual CPR to monthly SMM:
$$
SMM_t=1-(1-CPR_t)^{1/12}.
$$
Step 7: calculate prepayment:
$$
PP_t=(B_{t-1}-SP_t)SMM_t.
$$
Step 8: calculate total principal and ending balance:
$$
TP_t=SP_t+PP_t,
$$
$$
B_t=B_{t-1}-TP_t.
$$
For the PO and IO securities:
$$
\text{PO cash flow}_t=TP_t,
$$
$$
\text{IO cash flow}_t=IO_t.
$$


In [6]:

def level_payment(principal, monthly_rate, months):
    if months <= 0:
        return principal
    if abs(monthly_rate) < 1e-15:
        return principal / months
    return principal * monthly_rate / (1 - (1 + monthly_rate) ** (-months))


def cpr_to_smm(cpr):
    return 1 - (1 - cpr) ** (1 / 12)


def psa_cpr(month, seasoning_months=0, psa_multiple=1.0):
    age = month + seasoning_months
    base_cpr = 0.06 * min(age / 30, 1.0)
    return psa_multiple * base_cpr


def pass_through_cashflows(
    principal=400.0,
    mortgage_annual_rate=0.06,
    pass_through_annual_rate=0.05,
    term_months=240,
    seasoning_months=0,
    psa_multiple=1.0,
):
    """Excel-style re-amortizing pass-through cash flows.

    Units are millions of dollars.
    """
    balance = principal
    mortgage_monthly_rate = mortgage_annual_rate / 12
    pass_through_monthly_rate = pass_through_annual_rate / 12
    rows = []

    for month in range(1, term_months + 1):
        beginning_balance = balance
        remaining_months = term_months - month + 1

        monthly_payment = level_payment(beginning_balance, mortgage_monthly_rate, remaining_months)
        mortgage_interest = beginning_balance * mortgage_monthly_rate
        investor_interest = beginning_balance * pass_through_monthly_rate
        scheduled_principal = monthly_payment - mortgage_interest

        cpr = psa_cpr(month, seasoning_months, psa_multiple)
        smm = cpr_to_smm(cpr)
        prepayment = (beginning_balance - scheduled_principal) * smm

        total_principal = scheduled_principal + prepayment
        ending_balance = beginning_balance - total_principal

        rows.append({
            "month": month,
            "beginning_balance": beginning_balance,
            "investor_interest": investor_interest,
            "scheduled_principal": scheduled_principal,
            "prepayment": prepayment,
            "total_principal": total_principal,
            "ending_balance": ending_balance,
        })
        balance = ending_balance

    df = pd.DataFrame(rows)
    df.attrs["total_investor_interest"] = df["investor_interest"].sum()
    df.attrs["total_prepayments"] = df["prepayment"].sum()
    return df


pt_100 = pass_through_cashflows(psa_multiple=1.0)
pt_200 = pass_through_cashflows(psa_multiple=2.0)

print(f"Previous quiz check: investor interest at 100 PSA = {pt_100.attrs['total_investor_interest']:.2f} million")
print(f"Previous quiz check: prepayments at 100 PSA = {pt_100.attrs['total_prepayments']:.2f} million")
print(f"Previous quiz check: prepayments at 200 PSA = {pt_200.attrs['total_prepayments']:.2f} million")


Previous quiz check: investor interest at 100 PSA = 171.18 million
Previous quiz check: prepayments at 100 PSA = 181.09 million
Previous quiz check: prepayments at 200 PSA = 268.15 million



The checks above reproduce the accepted previous-quiz values:

| Quantity | Accepted value |
|---|---:|
| Total investor interest at 100 PSA | $171.18$ million |
| Total prepayments at 100 PSA | $181.09$ million |
| Total prepayments at 200 PSA | $268.15$ million |

This confirms that the cash-flow engine matches the convention used in the earlier mortgage pass-through questions.



## Questions 5--6: PO and IO present values

For an annual risk-free rate $r_f$, monthly compounding gives the discount factor:
$$
DF_t=\frac{1}{(1+r_f/12)^t}.
$$
### Question 5: PO MBS

The PO receives only principal. Therefore:
$$
PV_{PO}=\sum_{t=1}^{240}TP_tDF_t.
$$
For Question 5, use:
$$
r_f=4.5\%, \qquad \text{PSA}=100.
$$
### Question 6: IO MBS

The IO receives only pass-through interest. Therefore:
$$
PV_{IO}=\sum_{t=1}^{240}IO_tDF_t.
$$
For Question 6, use the same assumptions:
$$
r_f=4.5\%, \qquad \text{PSA}=100.
$$



## Question 7: IO average life

A normal principal average life weights time by principal cash flow. But an IO security has no principal cash flow.

So for this question, we weight by IO interest cash flows:
$$
\text{Average Life}_{IO}
=
\frac{\sum_{t=1}^{240}\left(\frac{t}{12}\right)IO_t}
{\sum_{t=1}^{240}IO_t}.
$$
This answers the question: on average, when are the IO interest cash flows received?



## Questions 8--9: IO profit/loss

Question 8 says the IO was purchased at the Question 6 price. Therefore, the purchase price is
$$
\text{Purchase Price}=PV_{IO}(r_f=4.5\%,100\text{ PSA}).
$$
If rates fall to $3.5\%$, the new IO value is
$$
PV_{IO}(r_f=3.5\%,100\text{ PSA}).
$$
So profit/loss is
$$
\text{P/L}_{Q8}
=PV_{IO}(3.5\%,100\text{ PSA})-PV_{IO}(4.5\%,100\text{ PSA}).
$$
Question 9 changes both assumptions:

- risk-free rate decreases from $4.5\%$ to $3.5\%$
- prepayment speed increases from $100$ PSA to $150$ PSA

So profit/loss is
$$
\text{P/L}_{Q9}
=PV_{IO}(3.5\%,150\text{ PSA})-PV_{IO}(4.5\%,100\text{ PSA}).
$$
Important intuition:

- A lower discount rate increases the present value of IO cash flows.
- A higher prepayment speed reduces future balances faster.
- Lower balances reduce future IO interest cash flows.

For Question 9, the faster prepayment effect dominates, causing a loss.


In [7]:

def value_po_io(psa_multiple=1.0, annual_rf_rate=0.045):
    cf = pass_through_cashflows(psa_multiple=psa_multiple)
    discount = 1 / (1 + annual_rf_rate / 12) ** cf["month"]

    po_pv = (cf["total_principal"] * discount).sum()
    io_pv = (cf["investor_interest"] * discount).sum()
    io_average_life = (cf["month"] * cf["investor_interest"]).sum() / cf["investor_interest"].sum() / 12

    return po_pv, io_pv, io_average_life, cf


po_100_45, io_100_45, io_avg_life, cf_100 = value_po_io(psa_multiple=1.0, annual_rf_rate=0.045)
_, io_100_35, _, _ = value_po_io(psa_multiple=1.0, annual_rf_rate=0.035)
_, io_150_35, _, _ = value_po_io(psa_multiple=1.5, annual_rf_rate=0.035)

q5 = po_100_45
q6 = io_100_45
q7 = io_avg_life
q8 = io_100_35 - io_100_45
q9 = io_150_35 - io_100_45

print(f"Q5 PO PV = {q5:.2f} million")
print(f"Q6 IO PV = {q6:.2f} million")
print(f"Q7 IO average life = {q7:.2f} years")
print(f"Q8 IO P/L from rf 4.5% to 3.5% = {q8:.2f} million")
print(f"Q9 IO P/L from rf 3.5% and 150 PSA = {q9:.2f} million")


Q5 PO PV = 280.10 million
Q6 IO PV = 133.23 million
Q7 IO average life = 6.01 years
Q8 IO P/L from rf 4.5% to 3.5% = 7.17 million
Q9 IO P/L from rf 3.5% and 150 PSA = -9.58 million



### Questions 5--9 interpretation

For Question 5:
$$
PV_{PO}=280.10\text{ million}.
$$
For Question 6:
$$
PV_{IO}=133.23\text{ million}.
$$
For Question 7:
$$
\text{Average Life}_{IO}=6.01\text{ years}.
$$
For Question 8:
$$
\text{P/L}=140.40-133.23=7.17\text{ million}.
$$
The IO gains value when the discount rate falls from $4.5\%$ to $3.5\%$, assuming prepayment stays at $100$ PSA.

For Question 9:
$$
\text{P/L}=123.65-133.23=-9.58\text{ million}.
$$
Even though the discount rate decreases, prepayment increases to $150$ PSA. Faster prepayment reduces outstanding balance faster, which reduces future IO interest cash flows. That is why the IO position loses money.



## Final answers to submit

| Question | Answer to submit |
|---|---:|
| 1. Forward-starting swap value | 33,374 |
| 2. Swaption value | 26,311 |
| 3. Hazard rate at time 0 (%) | 2.10 |
| 4. 5-year CDS par spread (bps) | 301.51 |
| 5. PO MBS present value (millions) | 280.10 |
| 6. IO MBS present value (millions) | 133.23 |
| 7. IO MBS average life (years) | 6.01 |
| 8. IO P/L, rate 4.5% to 3.5% (millions) | 7.17 |
| 9. IO P/L, rate 3.5% and 150 PSA (millions) | -9.58 |
